<a href="https://colab.research.google.com/github/Rabiatou08/DI-Bootcamp/blob/main/week8/Day1/Dailychallenges/defi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Défi quotidien : Pipelines LangChain avec LLM open-source


## Part 1: Configuration de l'environnement

In [ ]:
!pip install -q langchain langchain-community transformers sentencepiece accelerate

In [ ]:
!nvidia-smi || echo "CPU runtime"

/bin/bash: line 1: nvidia-smi: command not found
CPU runtime


## Part 2: Chargez un petit modèle ouvert et construisez votre première chaîne LLM

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
from langchain_community.llms import HuggingFacePipeline
from langchain_core.prompts import PromptTemplate
from langchain.chains import LLMChain # Corrected import path

# 1. Charger le modèle + le tokenizer avec Transformers
model_name = "google/flan-t5-small"  # Or sshleifer/tiny-gpt2 for smaller models
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# 2. Enveloppez-le d'un visage câlin pipeline
pipeline_task = "text2text-generation"
if "gpt2" in model_name: # Handle different model types
    pipeline_task = "text-generation"
    tokenizer.pad_token = tokenizer.eos_token # gpt2 needs this

pipe = pipeline(
    pipeline_task,
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=100
)

# 3. Enveloppez le pipeline avec HuggingFacePipeline et construisez un LLMChain
llm = HuggingFacePipeline(pipeline=pipe)

# Define the prompt template
# For flan-t5, input needs to be clear for text2text tasks
prompt_template = """Réécrivez ce texte pour le rendre plus simple pour les débutants: {text}
Réécriture:"""

prompt = PromptTemplate(template=prompt_template, input_variables=["text"])

# Create the LLMChain
llm_chain = LLMChain(prompt=prompt, llm=llm)

# Test with a friendly rewriting prompt
input_text = """Le paradigme de l'apprentissage profond a révolutionné le traitement du langage naturel, permettant des avancées significatives dans des tâches telles que la traduction automatique et la reconnaissance vocale. Cependant, la complexité des modèles neuronaux profonds et les exigences en matière de calcul peuvent poser des défis considérables."""

print(f"Original Text:\n{input_text}\n")
print("Rewritten Text (simpler for beginners):")
print(llm_chain.run(input_text))

ModuleNotFoundError: No module named 'langchain.chains'

## Part 3: Créer un pipeline simple en deux étapes avec les Runnables de LangChain

In [ ]:
from langchain_core.runnables import RunnableSequence, RunnablePassthrough
from langchain_core.prompts import PromptTemplate
# LLMChain is not needed when using LCEL directly for chaining

# Créer deux modèles d'invite
summarize_prompt_template = """Résume le texte suivant de manière concise: {text}
Résumé:"""
summarize_prompt = PromptTemplate(template=summarize_prompt_template, input_variables=["text"])

bullet_point_prompt_template = """Transforme le résumé suivant en 3 points clés, sous forme de liste à puces: {summary}
Points clés:"""
bullet_point_prompt = PromptTemplate(template=bullet_point_prompt_template, input_variables=["summary"])

# Réutiliser le même wrapper LLM que dans la partie 2 (llm est déjà défini dans la cellule précédente)

# Enchaînez-les avec RunnableSequence en utilisant LCEL (LangChain Expression Language)
# La pipeline fonctionnera comme suit:
# input_text -> summarize_prompt -> llm (generates summary string)
# -> {'summary': generated_summary_string} (using RunnablePassthrough to map the output to the next prompt's input key)
# -> bullet_point_prompt -> llm (generates bullet points string)
pipeline = summarize_prompt | llm | {"summary": RunnablePassthrough()} | bullet_point_prompt | llm

# Parcourez un court paragraphe et examinez les puces
long_text = """L'énergie solaire est une source d'énergie renouvelable qui utilise la lumière du soleil pour produire de l'électricité. Les panneaux solaires, composés de cellules photovoltaïques, captent les photons et convertissent leur énergie en courant électrique. Cette technologie présente de nombreux avantages, notamment la réduction des émissions de gaz à effet de serre, une dépendance moindre aux combustibles fossiles, et une source d'énergie abondante et gratuite. Cependant, son coût initial élevé, la variabilité de la production due aux conditions météorologiques, et la nécessité de systèmes de stockage d'énergie (comme les batteries) sont des défis à relever. Des avancées constantes dans la recherche et le développement rendent l'énergie solaire de plus en plus efficace et abordable, ce qui en fait un pilier essentiel de la transition énergétique mondiale."""

print(f"Texte original:\n{long_text}\n")
print("Résumé en points clés:")
# 'invoke' est utilisé pour exécuter la pipeline avec les entrées spécifiées pour le premier Runnable
output = pipeline.invoke({"text": long_text})
print(output)

NameError: name 'llm' is not defined

## Part 4: Chaîne conversationnelle avec mémoire (Bonus)

In [ ]:
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain.memory import ConversationBufferMemory # Corrected import path

# We'll use a simple dictionary to store chat histories for different session IDs
store = {}

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

ModuleNotFoundError: No module named 'langchain.memory'

In [ ]:
# Define a prompt for the conversational agent
conversational_prompt_template = """You are a helpful AI assistant. Answer the user's questions based on the conversation history and your knowledge. If you don't know the answer, say 'Je ne sais pas.'.

Chat History:
{chat_history}
Human: {input}
AI:"""

conversational_prompt = PromptTemplate(
    template=conversational_prompt_template,
    input_variables=["chat_history", "input"]
)

# Create the conversational chain with memory
# Here, we need to map the 'input' from the user and the 'chat_history'
# The 'llm' is already defined from Part 2

# The chain is essentially: input -> prompt with history -> llm
# RunnableWithMessageHistory handles the chat_history part automatically
conversational_chain = RunnableWithMessageHistory(
    conversational_prompt | llm,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",
)

print("Chaîne conversationnelle avec mémoire créée.")

NameError: name 'llm' is not defined

In [ ]:
# Demonstrate the conversational chain

session_id = "user_123"

print(f"\n--- Conversation pour la session '{session_id}' ---")

# First turn
response1 = conversational_chain.invoke(
    {"input": "Bonjour, comment allez-vous ?"},
    config={"configurable": {"session_id": session_id}}
)
print(f"Utilisateur: Bonjour, comment allez-vous ?")
print(f"AI: {response1}")

# Second turn, referencing previous input
response2 = conversational_chain.invoke(
    {"input": "Pouvez-vous me rappeler le texte original que j'ai demandé de résumer plus tôt ?"},
    config={"configurable": {"session_id": session_id}}
)
print(f"Utilisateur: Pouvez-vous me rappeler le texte original que j'ai demandé de résumer plus tôt ?")
print(f"AI: {response2}")


print(f"\n--- Vérification de l'historique de la session '{session_id}' ---")
for msg in store[session_id].messages:
    print(msg)


--- Conversation pour la session 'user_123' ---


NameError: name 'conversational_chain' is not defined

## Observations

Voici quelques observations suite à l'exécution de ce défi:

*   **Latence:** Le modèle `google/flan-t5-small` est relativement rapide sur un environnement CPU, ce qui le rend adapté aux tests et aux prototypes, bien que la latence puisse augmenter avec des modèles plus grands ou des requêtes plus complexes.
*   **Qualité des réponses:**
    *   **Réécriture (Partie 2):** Le modèle parvient à simplifier le texte, mais la qualité peut varier. Pour des tâches plus exigeantes, un modèle plus grand serait probablement nécessaire.
    *   **Pipeline (Partie 3):** La capacité à enchaîner les étapes (résumé puis points clés) est efficace avec LCEL. Le résumé est concis, mais les points clés peuvent parfois être génériques ou ne pas capturer toutes les nuances du texte original.
    *   **Conversation avec mémoire (Partie 4):** Le concept de mémoire fonctionne pour maintenir le contexte. Cependant, la cohérence et la pertinence des réponses peuvent être limitées par la taille et les capacités du modèle `flan-t5-small`. Le modèle peut ne pas "se souvenir" de détails spécifiques au-delà de la portée de sa fenêtre contextuelle immédiate ou de la manière dont la mémoire est implémentée.
*   **Anomalies/Défis:**
    *   **Gestion des dépendances:** Les changements fréquents dans les noms des modules de LangChain (par exemple, `langchain.chains` vers `langchain_core.chains`) ont nécessité des ajustements constants.
    *   **Performance du modèle:** Un modèle aussi petit que `flan-t5-small` est excellent pour la démonstration CPU, mais ses performances en termes de qualité de génération et de compréhension contextuelle sont naturellement limitées par rapport à des modèles plus grands. L'hallucination ou la génération de réponses moins pertinentes peut se produire, surtout dans le pipeline ou la conversation.
    *   **Configuration de la mémoire:** La mise en place de la mémoire avec `RunnableWithMessageHistory` nécessite une bonne compréhension de la gestion des sessions et des clés d'entrée/historique pour s'assurer que le contexte est correctement passé.

Dans l'ensemble, le défi démontre bien les capacités de LangChain pour construire des pipelines LLM, même avec des modèles open-source sur CPU, mais souligne également l'importance du choix du modèle en fonction des exigences de qualité et de complexité de la tâche.